El objetivo de este Notebook es construir un grafo multimodal de dominios a partir de:

Infraestructura (WHOIS/IP)

Contenido (TF-IDF/SBERT)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
cache  data  models  notebooks	README.md  requirements.txt  results


In [ ]:
import pandas as pd
import json
import numpy as np

import networkx as nx
from pathlib import Path


# Rutas del proyecto

In [ ]:
DATA_DIR = Path("data")   # Aquí se guarda los datos de entrada
RESULTS_DIR = Path("results")   # Aquí las salidas del análisis

DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_DIR / "fnn_processed.csv")
df.head()


,id,news_url,domain_norm,label,source_dataset
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,speedtalk.com,1,politifact
1,politifact15156,politics2020.info/index.php/2018/03/13/court-o...,politics2020.info,1,politifact
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467...,nscdscamps.org,1,politifact
3,politifact14355,https://howafrica.com/oscar-pistorius-attempts...,howafrica.com,1,politifact
4,politifact15371,http://washingtonsources.org/trump-votes-for-d...,washingtonsources.org,1,politifact


# Construcción de tabla de nodos (dominios)

In [ ]:
nodes = (
    df.groupby("domain_norm")   # Agrupo las noticias que pertenecen al mismo dominio
      .agg(
          n_news=("id", "count"),    # Cuántas noticias por dominio
          fake_count=("label", lambda x: (x == "fake").sum()),
          real_count=("label", lambda x: (x == "real").sum()),
          source_datasets=("source_dataset", lambda x: sorted(set(x)))   # De qué dataset sale el dominio
      )
      .reset_index()  # Desnormalizar
      .rename(columns={"domain_norm": "domain"})
)

nodes["fake_ratio"] = nodes["fake_count"] / nodes["n_news"]
nodes.head()


,domain,n_news,fake_count,real_count,source_datasets,fake_ratio
0,1001.com.do,1,0,0,[gossipcop],0.0
1,100percentfedup.com,1,0,0,[politifact],0.0
2,101kgb.iheart.com,1,0,0,[gossipcop],0.0
3,1029now.iheart.com,1,0,0,[gossipcop],0.0
4,1037theq.iheart.com,3,0,0,[gossipcop],0.0


In [ ]:
# Guardo la tabla de nodos
nodes.to_csv(RESULTS_DIR / "nodes_domains.csv", index=False)


# Aristas de infraestructura (WHOIS/IP)

In [ ]:
infra_edges = pd.read_csv(RESULTS_DIR / "whois_edges.csv")   # Cargo el CSV del Notebook 07
infra_edges.head()


,source,target,edge_type,evidence
0,101kgb.iheart.com,1029now.iheart.com,same_ip,199.232.210.193
1,101kgb.iheart.com,1037theq.iheart.com,same_ip,199.232.210.193
2,101kgb.iheart.com,1043myfm.iheart.com,same_ip,199.232.210.193
3,101kgb.iheart.com,1061kissfm.iheart.com,same_ip,199.232.210.193
4,101kgb.iheart.com,939litefm.iheart.com,same_ip,199.232.210.193


In [ ]:
INFRA_WEIGHTS = {
    "same_ip": 1.0,   # En la estructura pongo que si tienen la misma IP tienen una vinculación fuerte
    "same_registrar": 0.6    # Si tienen el mismo registrador (empresa) es una relación más débil
}

infra_edges["layer"] = "infra"    # Estas aristas las pongo que pertenecen a la capa de infraestructura
infra_edges["weight"] = infra_edges["edge_type"].map(INFRA_WEIGHTS)    # Pongo la estrucutura de infra_weights en las filas
infra_edges.head()


,source,target,edge_type,evidence,layer,weight
0,101kgb.iheart.com,1029now.iheart.com,same_ip,199.232.210.193,infra,1.0
1,101kgb.iheart.com,1037theq.iheart.com,same_ip,199.232.210.193,infra,1.0
2,101kgb.iheart.com,1043myfm.iheart.com,same_ip,199.232.210.193,infra,1.0
3,101kgb.iheart.com,1061kissfm.iheart.com,same_ip,199.232.210.193,infra,1.0
4,101kgb.iheart.com,939litefm.iheart.com,same_ip,199.232.210.193,infra,1.0


# Aristas de contenido (TF-IDF/SBERT)

In [ ]:
df_tfidf.columns  #Compruebo las columnas que hay en TF-IDF


Index(['domain_A', 'domain_B', 'similarity', 'edge_type', 'layer', 'weight'], dtype='object')

In [ ]:
df_sbert.columns  #Compruebo las columnas que hay en SBERT


Index(['source', 'target', 'weight', 'edge_type', 'layer'], dtype='object')

In [ ]:
content_edges = []

# TF-IDF
tfidf_path = RESULTS_DIR / "content_edges_tfidf.csv"
if tfidf_path.exists():
    df_tfidf = pd.read_csv(tfidf_path)
    df_tfidf["source"] = df_tfidf["domain_A"]
    df_tfidf["target"] = df_tfidf["domain_B"]
    df_tfidf["edge_type"] = "tfidf"
    df_tfidf["layer"] = "content"    # Estas aristas las pongo que pertenecen a la capa de contenido
    df_tfidf["weight"] = df_tfidf["similarity"]
    df_tfidf = df_tfidf[["source", "target", "edge_type", "layer", "weight"]]
    content_edges.append(df_tfidf)

# SBERT
sbert_path = RESULTS_DIR / "content_edges_sbert.csv"
if sbert_path.exists():
    df_sbert = pd.read_csv(sbert_path)
    df_sbert["edge_type"] = "sbert"
    df_sbert["layer"] = "content"    # Estas aristas las pongo que pertenecen a la capa de contenido
    df_sbert = df_sbert[["source", "target", "edge_type", "layer", "weight"]]     # En sbert weight no lo toco porque ya existe
    content_edges.append(df_sbert)


content_edges = pd.concat(content_edges, ignore_index=True)
content_edges.head()


,source,target,edge_type,layer,weight
0,US_News,politicsNews,tfidf,content,0.459242
1,US_News,News,tfidf,content,0.473685
2,US_News,Government News,tfidf,content,0.530405
3,US_News,left-news,tfidf,content,0.532447
4,US_News,worldnews,tfidf,content,0.419654


# Filtrado de aristas

In [ ]:
def apply_similarity_threshold(df, base_thr=0.10, percentile=97):
    thr_auto = np.percentile(df["weight"], percentile)    # Filtro para que me quede con las conexiones más fuertes (con el 3% superior)
    thr = max(base_thr, thr_auto)
    return df[df["weight"] >= thr], thr  # Nos quedamos con las aristas del 3% superior


In [ ]:
content_edges, final_thr = apply_similarity_threshold(content_edges)
final_thr


np.float64(0.9947145503157477)

# Unificar ambas aristas

In [ ]:
edges = pd.concat([infra_edges, content_edges], ignore_index=True)   # Uno aristas de infraestructura y contenido

# Canonicalizar pares (u < v)
# El grafo tiene que cumplir A-B = B-A (evito que haya duplicados independientemente si vienen de infraestructura o de contendio )
edges["u"] = edges[["source", "target"]].min(axis=1)   # Ordeno alfabéticamente ambos dominios (u va el primero)
edges["v"] = edges[["source", "target"]].max(axis=1)

edges = edges.drop(columns=["source", "target"])   # Borro las columnas que están al revés
edges = edges.rename(columns={"u": "source", "v": "target"})

edges.head()


,edge_type,evidence,layer,weight,source,target
0,same_ip,199.232.210.193,infra,1.0,101kgb.iheart.com,1029now.iheart.com
1,same_ip,199.232.210.193,infra,1.0,101kgb.iheart.com,1037theq.iheart.com
2,same_ip,199.232.210.193,infra,1.0,101kgb.iheart.com,1043myfm.iheart.com
3,same_ip,199.232.210.193,infra,1.0,101kgb.iheart.com,1061kissfm.iheart.com
4,same_ip,199.232.210.193,infra,1.0,101kgb.iheart.com,939litefm.iheart.com


In [ ]:
# Guardado del CSV final de aristas
edges.to_csv(RESULTS_DIR / "edges_multimodal.csv", index=False)


# Construcción grafo multimodal

In [ ]:
G = nx.Graph()   # Creo el grafo

for _, row in nodes.iterrows():
    G.add_node(        # Los nodos del grafo van a tener estos atributos
        row["domain"],
        n_news=int(row["n_news"]),
        fake_ratio=float(row["fake_ratio"]),
        source_datasets=",".join(row["source_datasets"])
    )



In [ ]:
for (u, v), group in edges.groupby(["source", "target"]):
    attrs = {       # Atributos de las aristas
        "has_same_ip": False,
        "has_same_registrar": False,
        "tfidf": 0.0,
        "sbert": 0.0
    }

    infra_weight = 0.0    # Fuerza de la infraestructura
    content_weight = 0.0   # Fuerza de contenido

    for _, r in group.iterrows():
        if r["edge_type"] == "same_ip":    # Si los dominios comparten misma IP
            attrs["has_same_ip"] = True
            infra_weight = max(infra_weight, r["weight"])
        elif r["edge_type"] == "same_registrar":   # Si los dominios comparten mismo registrador
            attrs["has_same_registrar"] = True
            infra_weight = max(infra_weight, r["weight"])
        elif r["edge_type"] == "tfidf":
            attrs["tfidf"] = max(attrs["tfidf"], r["weight"])    # Se guarda el mayor TF-IDF
            content_weight = max(content_weight, r["weight"])
        elif r["edge_type"] == "sbert":
            attrs["sbert"] = max(attrs["sbert"], r["weight"])    # Se guarda el mayor SBERT
            content_weight = max(content_weight, r["weight"])

    combined_weight = 1.2 * infra_weight + 1.0 * content_weight   # Le doy los siguientes pesos porque la infraestructura es más fiable que el contenido
    attrs["weight"] = combined_weight

    G.add_edge(u, v, **attrs)  # Añade una arista con estos atributos (** es desempaquetar los attrs de las aristas)


# Guardado de resultados (grafo final)

In [ ]:
nx.write_graphml(G, RESULTS_DIR / "graph_multimodal.graphml")  # Guardado de grafo en disco (como si fuese un PDF)


In [ ]:
import pickle

with open(RESULTS_DIR / "graph_multimodal.gpickle", "wb") as f:   # Guardado de grafo en formato binario (para seguir trabajando con el)
    pickle.dump(G, f)


In [ ]:
summary = {
    "n_nodes": G.number_of_nodes(),
    "n_edges": G.number_of_edges(),
    "edges_same_ip": sum(1 for _,_,d in G.edges(data=True) if d["has_same_ip"]),
    "edges_same_registrar": sum(1 for _,_,d in G.edges(data=True) if d["has_same_registrar"]),
    "edges_with_content": sum(1 for _,_,d in G.edges(data=True) if max(d["tfidf"], d["sbert"]) > 0),
    "connected_components": [len(c) for c in sorted(nx.connected_components(G), key=len, reverse=True)[:10]]
}

summary


{'n_nodes': 2431,
 'n_edges': 3875,
 'edges_same_ip': 3874,
 'edges_same_registrar': 0,
 'edges_with_content': 1,
 'connected_components': [49, 28, 21, 20, 18, 15, 14, 13, 13, 12]}

Mirando summary, se puede comprobar que la infraestructura de red constituye la principal fuente de conexión entre los dominios. Mientras que la similitud de contenido se utiliza como una señal complementaria (solo aparecen las similitudes de contenido extremadamnete altas).

In [ ]:
with open(RESULTS_DIR / "graph_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
